# 🚖 Rapido Ride Data Analysis
**Internship Project | Data Analytics**

**Dataset:** `rides_data.csv` — 50,000 Rapido ride records  
**Goal:** Analyze ride patterns, revenue trends, and derive business insights

---

## 📦 Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os

warnings.filterwarnings('ignore')

# Chart style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Output directory for saved images
IMG_DIR = '../report_images'
os.makedirs(IMG_DIR, exist_ok=True)

print('Libraries loaded successfully ✅')

---
## 📂 Step 2: Load Dataset

In [ ]:
df = pd.read_csv('../data/rides_data.csv')

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('\nColumn names:', df.columns.tolist())
df.head(5)

---
## 🧹 Step 3: Data Cleaning

In [ ]:
# ── 3.1 Initial data types & missing values ───────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

In [ ]:
# ── 3.2 Fix data types ────────────────────────────────────────────────────────
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['time'] = pd.to_datetime(df['time'], format='%H:%M:%S.%f', errors='coerce').dt.time

# Numeric columns — coerce to float (handles 'nan' strings)
for col in ['ride_charge', 'misc_charge', 'total_fare', 'distance', 'duration']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# payment_method — fill cancelled rides with 'N/A'
df['payment_method'] = df['payment_method'].fillna('N/A')

# Remove full duplicate rows
before = len(df)
df = df.drop_duplicates(subset='ride_id', keep='first')
print(f'Duplicates removed: {before - len(df)}')

print('\nCleaned data types:')
print(df.dtypes)

---
## 🛠️ Step 4: Feature Engineering

In [ ]:
# ── 4.1 Date-based features ───────────────────────────────────────────────────
df['year']       = df['date'].dt.year
df['month']      = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%b')
df['day_of_week']= df['date'].dt.day_name()
df['week']       = df['date'].dt.isocalendar().week.astype(int)

# ── 4.2 Hour of day ──────────────────────────────────────────────────────────
import datetime
def get_hour(t):
    if isinstance(t, datetime.time):
        return t.hour
    return np.nan

df['hour'] = df['time'].apply(get_hour)

# ── 4.3 Time of Day segment ──────────────────────────────────────────────────
def time_of_day(h):
    if pd.isna(h):     return 'Unknown'
    if 5 <= h < 12:    return 'Morning'
    if 12 <= h < 17:   return 'Afternoon'
    if 17 <= h < 21:   return 'Evening'
    return 'Night'

df['time_of_day'] = df['hour'].apply(time_of_day)

# ── 4.4 Revenue column (alias for completed rides) ───────────────────────────
df['revenue'] = np.where(df['ride_status'] == 'completed', df['total_fare'], 0)

# ── 4.5 City (source city — first two words) ─────────────────────────────────
df['source_area'] = df['source'].str.split().str[:2].str.join(' ')

print('New features created:')
print(df[['date','hour','time_of_day','revenue','source_area']].head(5))

---
## 📊 Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# ── 5.1 Basic Statistics ──────────────────────────────────────────────────────
completed = df[df['ride_status'] == 'completed']
cancelled = df[df['ride_status'] == 'cancelled']

total_rides    = len(df)
total_revenue  = completed['total_fare'].sum()
avg_fare       = completed['total_fare'].mean()
cancellation_rate = len(cancelled) / total_rides * 100
avg_distance   = completed['distance'].mean()
avg_duration   = completed['duration'].mean()

print('=' * 45)
print('           📌 KEY BUSINESS KPIs')
print('=' * 45)
print(f'  Total Rides          : {total_rides:>10,}')
print(f'  Completed Rides      : {len(completed):>10,}')
print(f'  Cancelled Rides      : {len(cancelled):>10,}')
print(f'  Total Revenue        : ₹{total_revenue:>10,.0f}')
print(f'  Average Fare         : ₹{avg_fare:>10.2f}')
print(f'  Cancellation Rate    : {cancellation_rate:>9.1f}%')
print(f'  Avg Distance (km)    : {avg_distance:>10.2f}')
print(f'  Avg Duration (min)   : {avg_duration:>10.1f}')
print('=' * 45)

In [ ]:
# ── 5.2 Rides by Service Type ─────────────────────────────────────────────────
service_counts = df['services'].value_counts().reset_index()
service_counts.columns = ['Service', 'Count']
print(service_counts)

In [ ]:
# ── 5.3 Payment Method Breakdown ─────────────────────────────────────────────
payment = df[df['ride_status']=='completed']['payment_method'].value_counts()
print('Payment Methods (Completed Rides):')
print(payment)

---
## 📈 Step 6: Visualizations

In [ ]:
# ── Chart 1: KPI Summary Bar ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Executive KPI Overview', fontsize=15, fontweight='bold', y=1.02)

kpi_labels = ['Total Rides', 'Total Revenue (₹)', 'Avg Fare (₹)']
kpi_values = [total_rides, total_revenue, avg_fare]
kpi_colors = ['#3b82d4', '#22c55e', '#f59e0b']

for ax, label, val, color in zip(axes, kpi_labels, kpi_values, kpi_colors):
    ax.bar([label], [val], color=color, edgecolor='white', width=0.4)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_ylabel('')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.tick_params(axis='x', labelbottom=False)
    ax.text(0, val * 0.5, f'{val:,.1f}', ha='center', va='center',
            fontsize=14, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/01_kpi_overview.png', bbox_inches='tight')
plt.show()
print('Chart saved ✅')

In [ ]:
# ── Chart 2: Ride Status Distribution ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Ride Status Distribution', fontsize=14, fontweight='bold')

status_counts = df['ride_status'].value_counts()
colors = ['#22c55e', '#ef4444', '#f59e0b', '#3b82d4']

ax1.pie(status_counts.values, labels=status_counts.index,
        autopct='%1.1f%%', colors=colors[:len(status_counts)],
        startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax1.set_title('By Status (Pie)', fontsize=12)

status_counts.plot(kind='bar', ax=ax2, color=colors[:len(status_counts)],
                   edgecolor='white')
ax2.set_title('By Status (Bar)', fontsize=12)
ax2.set_xlabel('Ride Status')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=0)
for p in ax2.patches:
    ax2.annotate(f'{p.get_height():,.0f}', (p.get_x()+p.get_width()/2, p.get_height()),
                 ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/02_ride_status.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 3: Revenue by Service Type ─────────────────────────────────────────
rev_by_service = completed.groupby('services')['total_fare'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = rev_by_service.plot(kind='bar', ax=ax, color=['#3b82d4','#22c55e','#f59e0b'],
                            edgecolor='white')
ax.set_title('Total Revenue by Service Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Service Type')
ax.set_ylabel('Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e6:.1f}M'))
ax.tick_params(axis='x', rotation=0)
for p in ax.patches:
    ax.annotate(f'₹{p.get_height()/1e6:.2f}M', (p.get_x()+p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/03_revenue_by_service.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 4: Monthly Revenue Trend ───────────────────────────────────────────
monthly = completed.groupby(['year','month','month_name'])['total_fare'].sum().reset_index()
monthly = monthly.sort_values(['year','month'])
monthly['period'] = monthly['month_name'] + ' ' + monthly['year'].astype(str)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly['period'], monthly['total_fare'], marker='o', linewidth=2.5,
        color='#3b82d4', markersize=6)
ax.fill_between(range(len(monthly)), monthly['total_fare'],
                alpha=0.15, color='#3b82d4')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['period'], rotation=45, ha='right')
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1e6:.1f}M'))

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/04_monthly_revenue.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 5: Peak Hours Analysis ─────────────────────────────────────────────
hourly_rides = df.groupby('hour').size().reset_index(name='rides')

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(hourly_rides['hour'], hourly_rides['rides'],
       color=['#ef4444' if r == hourly_rides['rides'].max() else '#3b82d4'
              for r in hourly_rides['rides']],
       edgecolor='white')
ax.set_title('Rides by Hour of Day (Peak Hours)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour (0–23)')
ax.set_ylabel('Number of Rides')
ax.set_xticks(range(0, 24))
peak_hour = hourly_rides.loc[hourly_rides['rides'].idxmax(), 'hour']
ax.axvline(x=peak_hour, color='red', linestyle='--', alpha=0.5, label=f'Peak: {peak_hour}:00')
ax.legend()

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/05_peak_hours.png', bbox_inches='tight')
plt.show()
print(f'Peak hour: {peak_hour}:00')

In [ ]:
# ── Chart 6: Time of Day Distribution ────────────────────────────────────────
tod_order = ['Morning', 'Afternoon', 'Evening', 'Night']
tod_counts = df['time_of_day'].value_counts().reindex(tod_order)

fig, ax = plt.subplots(figsize=(8, 5))
colors_tod = ['#f59e0b', '#22c55e', '#f97316', '#6366f1']
bars = ax.bar(tod_counts.index, tod_counts.values, color=colors_tod, edgecolor='white')
ax.set_title('Rides by Time of Day', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Rides')
for bar, val in zip(bars, tod_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{val:,}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/06_time_of_day.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 7: Fare Distribution (Histogram) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(completed['total_fare'].dropna(), bins=50, color='#3b82d4',
        edgecolor='white', alpha=0.85)
ax.axvline(completed['total_fare'].mean(), color='red', linestyle='--',
           linewidth=2, label=f'Mean: ₹{completed["total_fare"].mean():.0f}')
ax.axvline(completed['total_fare'].median(), color='green', linestyle='--',
           linewidth=2, label=f'Median: ₹{completed["total_fare"].median():.0f}')
ax.set_title('Fare Distribution (Completed Rides)', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Fare (₹)')
ax.set_ylabel('Frequency')
ax.legend()

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/07_fare_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 8: Payment Method Analysis ─────────────────────────────────────────
pay_data = completed['payment_method'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Payment Method Analysis', fontsize=14, fontweight='bold')

pal = ['#3b82d4','#22c55e','#f59e0b','#ef4444','#8b5cf6']

ax1.pie(pay_data.values, labels=pay_data.index, autopct='%1.1f%%',
        colors=pal[:len(pay_data)],
        startangle=140, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
ax1.set_title('Share by Payment Method', fontsize=12)

pay_data.plot(kind='barh', ax=ax2, color=pal[:len(pay_data)], edgecolor='white')
ax2.set_title('Count by Payment Method', fontsize=12)
ax2.set_xlabel('Number of Rides')
for p in ax2.patches:
    ax2.annotate(f'{p.get_width():,.0f}', (p.get_width(), p.get_y()+p.get_height()/2),
                 ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/08_payment_methods.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 9: Top 10 Source Areas by Rides ────────────────────────────────────
top_areas = df['source_area'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 6))
top_areas.sort_values().plot(kind='barh', ax=ax, color='#3b82d4', edgecolor='white')
ax.set_title('Top 10 Source Areas by Rides', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Rides')
for p in ax.patches:
    ax.annotate(f'{p.get_width():,.0f}', (p.get_width(), p.get_y()+p.get_height()/2),
                ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/09_top_areas.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 10: Day-of-Week Heatmap (Rides by Hour) ─────────────────────────────
heatmap_data = df.groupby(['day_of_week', 'hour']).size().unstack(fill_value=0)
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
heatmap_data = heatmap_data.reindex(day_order)

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd', linewidths=0.4,
            cbar_kws={'label': 'Number of Rides'})
ax.set_title('Rides Heatmap: Day of Week × Hour', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Day of Week')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/10_heatmap_day_hour.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 11: Cancellation Rate by Service ───────────────────────────────────
cancel_by_service = df.groupby('services').apply(
    lambda x: (x['ride_status']=='cancelled').sum() / len(x) * 100
).sort_values(ascending=False).reset_index()
cancel_by_service.columns = ['Service', 'Cancellation Rate (%)']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(cancel_by_service['Service'],
              cancel_by_service['Cancellation Rate (%)'],
              color=['#ef4444','#f59e0b','#3b82d4'], edgecolor='white')
ax.set_title('Cancellation Rate by Service Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Cancellation Rate (%)')
ax.set_xlabel('Service')
ax.tick_params(axis='x', rotation=0)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/11_cancellation_by_service.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Chart 12: Distance vs Fare Scatter ───────────────────────────────────────
sample = completed.dropna(subset=['distance','total_fare']).sample(
    min(3000, len(completed)), random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
service_colors = {'cab economy': '#3b82d4', 'auto': '#22c55e', 'parcel': '#f59e0b'}
for svc, grp in sample.groupby('services'):
    ax.scatter(grp['distance'], grp['total_fare'],
               label=svc, alpha=0.4, s=20,
               color=service_colors.get(svc, 'gray'))
ax.set_title('Distance vs Total Fare by Service', fontsize=14, fontweight='bold')
ax.set_xlabel('Distance (km)')
ax.set_ylabel('Total Fare (₹)')
ax.legend(title='Service')

plt.tight_layout()
plt.savefig(f'{IMG_DIR}/12_distance_vs_fare.png', bbox_inches='tight')
plt.show()

---
## 🏆 Step 7: Business KPIs Summary

In [ ]:
# ── 7.1 Monthly Growth % ──────────────────────────────────────────────────────
monthly_rev = completed.groupby(['year','month'])['total_fare'].sum().reset_index()
monthly_rev = monthly_rev.sort_values(['year','month'])
monthly_rev['growth_pct'] = monthly_rev['total_fare'].pct_change() * 100

print('Monthly Revenue with Growth %:')
print(monthly_rev[['year','month','total_fare','growth_pct']].to_string(index=False))

In [ ]:
# ── 7.2 Full KPI Table ────────────────────────────────────────────────────────
kpi_data = {
    'KPI': [
        'Total Rides', 'Completed Rides', 'Cancelled Rides',
        'Total Revenue (₹)', 'Avg Fare per Ride (₹)',
        'Cancellation Rate (%)', 'Avg Distance (km)',
        'Avg Duration (min)', 'Most Used Payment', 'Peak Hour'
    ],
    'Value': [
        f'{total_rides:,}',
        f'{len(completed):,}',
        f'{len(cancelled):,}',
        f'₹{total_revenue:,.0f}',
        f'₹{avg_fare:.2f}',
        f'{cancellation_rate:.1f}%',
        f'{avg_distance:.2f} km',
        f'{avg_duration:.1f} min',
        completed['payment_method'].mode()[0],
        f'{peak_hour}:00'
    ]
}
kpi_df = pd.DataFrame(kpi_data)
print('\n📌 Full KPI Dashboard')
print(kpi_df.to_string(index=False))

---
## 💡 Step 8: Business Insights

In [ ]:
insights = """
╔══════════════════════════════════════════════════════════════════════╗
║              💡 BUSINESS INSIGHTS (Risk | Opportunity | Action)      ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. CANCELLATION RISK                                                ║
║     Risk       → High cancellation rates reduce revenue & trust.    ║
║     Opportunity→ Identify top cancellation hours & offer incentives. ║
║     Action     → Notify drivers in advance + add penalty for cancels.║
║                                                                      ║
║  2. PEAK HOURS OPPORTUNITY                                           ║
║     Risk       → Demand spikes lead to driver shortages.             ║
║     Opportunity→ Surge pricing can boost revenue during peak hours.  ║
║     Action     → Schedule more drivers 7–9 AM & 6–9 PM.             ║
║                                                                      ║
║  3. PAYMENT METHOD ADOPTION                                          ║
║     Risk       → Cash dominance increases fraud and reconciliation.  ║
║     Opportunity→ Push UPI/wallet adoption with cashback offers.      ║
║     Action     → Launch 5% cashback on GPay/Amazon Pay rides.        ║
║                                                                      ║
║  4. LOW-PERFORMING AREAS                                             ║
║     Risk       → Some areas generate very few rides.                 ║
║     Opportunity→ Target areas with high demand & low supply.         ║
║     Action     → Marketing campaigns + driver incentives in hotspots.║
║                                                                      ║
║  5. SERVICE MIX                                                      ║
║     Risk       → Over-reliance on one service type is fragile.       ║
║     Opportunity→ Parcel service is an emerging revenue stream.       ║
║     Action     → Promote parcel delivery with business partnerships. ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(insights)

---
## ✅ Step 9: Export Cleaned Data

In [ ]:
df.to_csv('../data/rides_data_cleaned.csv', index=False)
print('Cleaned dataset exported → data/rides_data_cleaned.csv ✅')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

---
*Analysis complete. All charts saved to `report_images/`. Run `streamlit run dashboard/app.py` to launch the interactive dashboard.*